## 1. Окружение и зависимости



In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

!pip install -q -U ultralytics transformers segmentation-models-pytorch timm einops kornia gradio==4.44.0

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

WEIGHTS_DIR = Path('/content/drive/MyDrive/Диплом/pipeline_weights')
MODELS_DIR  = Path('/content/drive/MyDrive/Диплом/models')

NASTYA_SCRIPT   = WEIGHTS_DIR / 'simple_isic_full_inference.py'
RTDETR_WEIGHTS  = WEIGHTS_DIR / 'detector_rtdetr_l_img256_best.pt'
HAIR_WEIGHTS    = WEIGHTS_DIR / 'best_weights_Hair_U-Net_320.pth'
PCONV_WEIGHTS   = WEIGHTS_DIR / 'best_weights_partial_conv_white_masked_input_img320.pth'
BIREFNET_WEIGHTS = WEIGHTS_DIR / 'best_model_birefnet_320.pt'
EXP6_WEIGHTS    = MODELS_DIR  / 'best_exp6.pth'

required = {
    'Скрипт Насти':       NASTYA_SCRIPT,
    'RT-DETR-L':          RTDETR_WEIGHTS,
    'Hair U-Net':         HAIR_WEIGHTS,
    'Partial Conv U-Net': PCONV_WEIGHTS,
    'BiRefNet':           BIREFNET_WEIGHTS,
    'Эксп.6': EXP6_WEIGHTS,
}
missing = []
for label, path in required.items():
    if path.exists():
        print(f'✓ {label:<24}  {path.stat().st_size/1e6:6.1f} МБ  {path.name}')
    else:
        print(f'✗ {label:<24}  НЕ НАЙДЕН: {path}')
        missing.append(label)

if missing:
    raise FileNotFoundError(
        f'Не хватает {len(missing)} файлов. Загрузи их в Drive и перезапусти ячейку.'
    )

In [ ]:
import sys
if str(WEIGHTS_DIR) not in sys.path:
    sys.path.insert(0, str(WEIGHTS_DIR))

from simple_isic_full_inference import (
    load_detector, load_hair_unet, load_pconv, load_birefnet,
    Detection, expand_and_clip_box,
    pil_to_tensor_01, imagenet_normalize, forward_logits,
    expand_binary_mask,
    IMG_SIZE, DETECTION_CONF, DETECTION_NMS_IOU, DETECTION_CROP_PAD,
    HAIR_THRESHOLD, LESION_THRESHOLD, RTDETR_DET_IMGSZ,
    DEVICE, DEVICE_ID,
)

import time

t0 = time.time(); detector = load_detector('rtdetr', RTDETR_WEIGHTS);   print(f'✓ RT-DETR-L  ({time.time()-t0:.1f}с)')
t0 = time.time(); hair_model = load_hair_unet(HAIR_WEIGHTS);            print(f'✓ Hair U-Net ({time.time()-t0:.1f}с)')
t0 = time.time(); pconv_model = load_pconv(PCONV_WEIGHTS);              print(f'✓ Partial Conv U-Net ({time.time()-t0:.1f}с)')
t0 = time.time(); birefnet = load_birefnet(BIREFNET_WEIGHTS);           print(f'✓ BiRefNet   ({time.time()-t0:.1f}с)')

if torch.cuda.is_available():
    print(f'\nVRAM: {torch.cuda.memory_allocated()/1e9:.2f} ГБ')

In [ ]:
import torch.nn as nn
import timm

class MultiHeadEfficientNet(nn.Module):
    """Архитектура из эксп.6: EfficientNet-B3 backbone + 7 голов признаков + 1 голова диагноза."""
    def __init__(self, n_features=7, in_channels=4, dropout=0.3):
        super().__init__()
        base = timm.create_model('efficientnet_b3', pretrained=False, num_classes=0)
        feat_dim = base.num_features
        old = base.conv_stem
        new = nn.Conv2d(in_channels, old.out_channels,
                        old.kernel_size, old.stride, old.padding,
                        bias=old.bias is not None)
        base.conv_stem = new
        self.backbone = base
        self.feature_heads = nn.ModuleList([
            nn.Sequential(nn.Dropout(dropout), nn.Linear(feat_dim, 1))
            for _ in range(n_features)
        ])
        self.diagnosis_head = nn.Sequential(nn.Dropout(dropout), nn.Linear(feat_dim, 1))

    def forward(self, x):
        f = self.backbone(x)
        feat_logits = torch.cat([h(f) for h in self.feature_heads], dim=1)
        diag_logit  = self.diagnosis_head(f)
        return feat_logits, diag_logit


model_exp6 = MultiHeadEfficientNet(n_features=7, in_channels=4, dropout=0.3).to(DEVICE)
state = torch.load(EXP6_WEIGHTS, map_location=DEVICE)
model_exp6.load_state_dict(state)
model_exp6.eval()
print('✓ Модель эксп.6 загружена')

if torch.cuda.is_available():
    print(f'VRAM после загрузки всех моделей: {torch.cuda.memory_allocated()/1e9:.2f} ГБ')

## Константы пайплайна

In [ ]:
import numpy as np

# Признаки эксп.6 (порядок голов 0-6)
FEATURES = [
    'pigment_network', 'streaks', 'pigmentation',
    'regression_structures', 'dots_and_globules',
    'blue_whitish_veil', 'vascular_structures'
]
FEAT_RU = [
    'Пигментная сеть', 'Полосы', 'Пигментация',
    'Регрессионные структуры', 'Точки и глобулы',
    'Бело-голубая вуаль', 'Сосудистые структуры'
]
ARGENZIANO_WEIGHTS = {
    'pigment_network': 2, 'streaks': 1, 'pigmentation': 1,
    'regression_structures': 1, 'dots_and_globules': 1,
    'blue_whitish_veil': 2, 'vascular_structures': 2,
}
ARGENZIANO_SCORES = [ARGENZIANO_WEIGHTS[f] for f in FEATURES]
SUSPICION_THRESHOLD = 3
COMMON_FEATURES_IDX = [0, 1, 4]

COMMON_MAX_SCORE = sum(ARGENZIANO_SCORES[i] for i in COMMON_FEATURES_IDX)  # = 4
COMMON_SUSPICION_THRESHOLD = 2 # упрощённый порог для 3-признаковой версии

# Размер входа модели
CLASSIFIER_IMG_SIZE = 300
CLASSIFIER_THRESHOLD = 0.4

# Параметры предобработки волос
HAIR_MASK_DILATION = 0


# Путь к весам VGG16 — лежит рядом с другими весами Насти
VGG16_WEIGHTS_PATH = '/content/drive/MyDrive/Диплом/pipeline_weights/best_model_vgg_320.pth'

# Размер входа VGG16
VGG16_IMG_SIZE = 320

# Порог уверенности для бинаризации выхода VGG16
VGG16_FEATURE_THRESHOLD = 0.5

# Минимальная площадь маски (пикселей), чтобы считать признак "обнаруженным"
VGG16_MIN_AREA_PX = 50

COMMON_FEATURES_IDX_MAP = {0: 0, 1: 2, 4: 4}

# Константы для подготовки входа VGG16
VGG16_IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
VGG16_IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
VGG16_LESION_DILATE_PX = 5
VGG16_NUM_CLASSES = 5
VGG16_IN_CHANNELS = 5 # RGB + B-канал LAB + маска невуса

print('Признаки:')
for f, ru, w in zip(FEATURES, FEAT_RU, ARGENZIANO_SCORES):
    print(f'  {ru:<26} {w}б')

In [ ]:
import cv2
from PIL import Image
import torch.nn.functional as F

@torch.no_grad()
def detect_lesion(image_pil: Image.Image, image_path: Path) -> Detection:
    """Шаг 1. RT-DETR на полноразмерном изображении.
    Если детектор ничего не нашёл - fallback: bbox = всё изображение.
    """
    width, height = image_pil.size
    results = detector.predict(
        source=str(image_path),
        imgsz=RTDETR_DET_IMGSZ,
        conf=DETECTION_CONF,
        iou=DETECTION_NMS_IOU,
        device=DEVICE_ID,
        verbose=False,
        half=torch.cuda.is_available(),
        max_det=20,
    )
    result = results[0]
    if result.boxes is None or len(result.boxes) == 0:
        return Detection((0, 0, width, height), 0.0, 'rtdetr', used_fallback=True)
    scores = result.boxes.conf.detach().cpu().float().numpy()
    best_idx = int(np.argmax(scores))
    raw_box = result.boxes.xyxy[best_idx].detach().cpu().float().numpy().tolist()
    box = expand_and_clip_box(tuple(raw_box), width, height, DETECTION_CROP_PAD)
    return Detection(box, float(scores[best_idx]), 'rtdetr', used_fallback=False)


@torch.no_grad()
def remove_hair_on_crop(crop: Image.Image) -> dict:
    image_01 = pil_to_tensor_01(crop, IMG_SIZE).to(DEVICE)
    image_norm = imagenet_normalize(image_01)

    # Hair U-Net
    hair_prob = torch.sigmoid(hair_model(image_norm))[0, 0].float()
    hair_mask_raw = (hair_prob > HAIR_THRESHOLD).float()
    hair_mask_np = (hair_mask_raw.cpu().numpy() * 255).astype(np.uint8)
    if HAIR_MASK_DILATION > 0:
        hair_mask_np = expand_binary_mask(hair_mask_np, HAIR_MASK_DILATION)
    hair_mask = torch.from_numpy((hair_mask_np > 127).astype(np.float32)
                                 ).unsqueeze(0).unsqueeze(0).to(DEVICE)

    # Partial Conv U-Net inpainting
    image_m11 = image_01 * 2.0 - 1.0
    x_masked = image_m11 * (1.0 - hair_mask) + hair_mask
    restored_m11 = pconv_model(x_masked, 1.0 - hair_mask)
    composite_m11 = restored_m11 * hair_mask + image_m11 * (1.0 - hair_mask)
    clean_01 = (composite_m11.clamp(-1, 1) + 1.0) / 2.0

    return {
        'crop_01': image_01[0].cpu(),
        'hair_mask': hair_mask[0, 0].cpu(),
        'clean_crop_01': clean_01[0].cpu(),
    }


@torch.no_grad()
def segment_lesion(clean_crop_01_cpu: torch.Tensor) -> torch.Tensor:
    """Шаг 4. BiRefNet на очищенном кропе → бинарная маска невуса (320×320)."""
    clean_crop_01 = clean_crop_01_cpu.unsqueeze(0).to(DEVICE)
    image_norm = imagenet_normalize(clean_crop_01)
    model_dtype = next(birefnet.parameters()).dtype
    logits = forward_logits(birefnet, image_norm.to(dtype=model_dtype),
                            target_hw=(IMG_SIZE, IMG_SIZE))
    prob = torch.sigmoid(logits.float())[0, 0]
    mask = (prob > LESION_THRESHOLD).float()
    return mask.cpu()


def prepare_classifier_input(crop_pil: Image.Image, lesion_mask_cpu: torch.Tensor) -> torch.Tensor:
    # RGB 300×300, нормализация ImageNet
    crop_resized = crop_pil.resize((CLASSIFIER_IMG_SIZE, CLASSIFIER_IMG_SIZE), Image.BILINEAR)
    arr = np.array(crop_resized, np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], np.float32)
    std  = np.array([0.229, 0.224, 0.225], np.float32)
    rgb_norm = (arr - mean) / std

    # Маска невуса 320×320 → 300×300 (NEAREST), нормализация (m-0.5)/0.5
    mask_np = lesion_mask_cpu.numpy()
    mask_resized = cv2.resize(mask_np, (CLASSIFIER_IMG_SIZE, CLASSIFIER_IMG_SIZE),
                              interpolation=cv2.INTER_NEAREST)
    mask_norm = (mask_resized - 0.5) / 0.5

    # Стек: (4, H, W)
    tensor = torch.from_numpy(np.concatenate(
        [rgb_norm, mask_norm[..., np.newaxis]], axis=2
    )).permute(2, 0, 1).float()
    return tensor.unsqueeze(0)  # (1, 4, 300, 300)


@torch.no_grad()
def classify_features(input_tensor: torch.Tensor) -> dict:
    x = input_tensor.to(DEVICE)
    feat_logits, diag_logit = model_exp6(x)
    feat_probs = torch.sigmoid(feat_logits)[0].cpu().numpy()
    diag_prob  = torch.sigmoid(diag_logit)[0, 0].cpu().item()
    return {
        'feat_probs': feat_probs,  # (7,)
        'feat_preds': (feat_probs >= CLASSIFIER_THRESHOLD).astype(int),
        'diag_prob':  diag_prob,
        'diag_pred':  int(diag_prob >= CLASSIFIER_THRESHOLD),
    }


def compute_argenziano_score(feat_preds: np.ndarray,
                              vgg_presence: np.ndarray = None) -> tuple[int, bool, list[str]]:
    """Балл с учётом OR-объединения по 3 общим признакам.

    feat_preds — бинарные предсказания классификационной модели (7 признаков).
    vgg_presence — бинарные предсказания VGG16-сегментации (5 признаков, опционально).

    Для 3 общих признаков применяется OR: признак считается обнаруженным,
    если его нашла хотя бы одна модель. Остальные 4 признака — только из классификатора.
    """
    combined = feat_preds.copy()
    if vgg_presence is not None:
        for my_i, vgg_i in COMMON_FEATURES_IDX_MAP.items():
            combined[my_i] = int(bool(feat_preds[my_i]) or bool(vgg_presence[vgg_i]))

    score = int(np.dot(combined, ARGENZIANO_SCORES))
    found = [FEAT_RU[i] for i, p in enumerate(combined) if p == 1]
    return score, score >= SUSPICION_THRESHOLD, found

print('✓ Функции пайплайна готовы')

## Сегментация признаков VGG16

In [ ]:
import os
import shutil
LOCAL_VGG_PATH = '/content/best_model_vgg_320.pth'
if not os.path.exists(LOCAL_VGG_PATH):
    shutil.copy(VGG16_WEIGHTS_PATH, LOCAL_VGG_PATH)
VGG16_WEIGHTS_PATH = LOCAL_VGG_PATH

In [ ]:
import torchvision.models as tv_models

class VGG16ArticleSeg(nn.Module):
    """Сегментационная VGG16-архитектура от А. Агафьиной (копия из её inference-скрипта)."""
    def __init__(self, num_classes=VGG16_NUM_CLASSES, in_channels=VGG16_IN_CHANNELS):
        super().__init__()
        base = tv_models.vgg16(weights=None)
        old_conv = base.features[0]
        new_conv = nn.Conv2d(
            in_channels, old_conv.out_channels,
            kernel_size=old_conv.kernel_size, stride=old_conv.stride,
            padding=old_conv.padding, dilation=old_conv.dilation,
            groups=old_conv.groups, bias=old_conv.bias is not None,
            padding_mode=old_conv.padding_mode,
        )
        with torch.no_grad():
            new_conv.weight[:, :3] = old_conv.weight
            mean_weight = old_conv.weight.mean(dim=1, keepdim=True)
            for c in range(3, in_channels):
                new_conv.weight[:, c:c+1] = mean_weight
            if old_conv.bias is not None:
                new_conv.bias.copy_(old_conv.bias)
        base.features[0] = new_conv
        self.features = base.features
        self.capture_ids = [3, 8, 15, 22, 27, 29]
        self.reduce4   = nn.Conv2d(512, 64, kernel_size=1)
        self.reduce5_2 = nn.Conv2d(512, 32, kernel_size=1)
        self.reduce5_3 = nn.Conv2d(512, 32, kernel_size=1)
        self.head = nn.Conv2d(576, num_classes, kernel_size=1)

    def forward(self, x):
        h, w = x.shape[-2:]
        feats = {}
        out = x
        for i, layer in enumerate(self.features):
            out = layer(out)
            if i in self.capture_ids:
                feats[i] = out
        ups = [feats[3], feats[8], feats[15],
               self.reduce4(feats[22]),
               self.reduce5_2(feats[27]),
               self.reduce5_3(feats[29])]
        ups = [F.interpolate(f, size=(h, w), mode="bilinear", align_corners=False)
               if f.shape[-2:] != (h, w) else f for f in ups]
        return self.head(torch.cat(ups, dim=1))


def load_vgg16_seg(weights_path):
    """Загрузка обученных весов. Пробуем разные форматы checkpoint."""
    model = VGG16ArticleSeg(num_classes=VGG16_NUM_CLASSES, in_channels=VGG16_IN_CHANNELS)
    ckpt = torch.load(weights_path, map_location='cpu')
    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        state = ckpt['model_state_dict']
    elif isinstance(ckpt, dict) and 'model_state' in ckpt:
        state = ckpt['model_state']
    elif isinstance(ckpt, dict) and 'state_dict' in ckpt:
        state = ckpt['state_dict']
    else:
        state = ckpt
    model.load_state_dict(state, strict=True)
    return model.to(DEVICE).eval()


vgg16_seg = load_vgg16_seg(VGG16_WEIGHTS_PATH)
print('✓ VGG16 (сегментация признаков от А. Агафьиной) загружена')


def dilate_binary_mask_np(mask_uint8, px):
    """Расширение бинарной маски на px пикселей (эллиптическое ядро)."""
    if px <= 0:
        return mask_uint8.astype(np.float32)
    k = 2 * px + 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    return cv2.dilate(mask_uint8.astype(np.uint8), kernel, iterations=1).astype(np.float32)


def segment_features_vgg16(original_crop_pil, lesion_mask_cpu):
    """Инференс VGG16-сегментации.

    На вход исходный кроп с волосами (original_crop_pil) и маска невуса от BiRefNet.
    На выход — попиксельные маски и карты вероятностей для 5 структур.
    """
    size = VGG16_IMG_SIZE

    # 1. Изображение → 320×320, RGB uint8 (с волосами!)
    crop_np = np.asarray(original_crop_pil.convert('RGB').resize((size, size), Image.BILINEAR))

    # 2. Маска невуса → 320×320 бинарная
    lesion_np = (lesion_mask_cpu.numpy() > 0.5).astype(np.uint8)
    if lesion_np.shape != (size, size):
        lesion_np = cv2.resize(lesion_np, (size, size), interpolation=cv2.INTER_NEAREST)

    # 3. Расширение маски (dilate 5 px)
    lesion_for_model = dilate_binary_mask_np(lesion_np, VGG16_LESION_DILATE_PX)

    # 4. Маскируем фон средним по ImageNet
    neutral_rgb = np.round(VGG16_IMAGENET_MEAN * 255.0).astype(np.uint8)
    crop_masked = crop_np.copy()
    crop_masked[~lesion_for_model.astype(bool)] = neutral_rgb

    # 5. Нормализация RGB
    rgb = crop_masked.astype(np.float32) / 255.0
    rgb_norm = (rgb - VGG16_IMAGENET_MEAN) / VGG16_IMAGENET_STD

    # 6. B-канал LAB
    lab = cv2.cvtColor(crop_masked, cv2.COLOR_RGB2LAB).astype(np.float32)
    b_norm = (lab[..., 2] - 128.0) / 64.0

    # 7. Собираем 5 каналов: R, G, B, B-LAB, mask
    channels = [rgb_norm[..., 0], rgb_norm[..., 1], rgb_norm[..., 2],
                b_norm, lesion_for_model.astype(np.float32)]
    x_np = np.ascontiguousarray(np.stack(channels, axis=0).astype(np.float32))
    x = torch.from_numpy(x_np).unsqueeze(0).to(DEVICE)

    # 8. Инференс
    with torch.no_grad():
        logits = vgg16_seg(x)
        lesion_t = torch.from_numpy(lesion_for_model[None, None].astype(np.float32)).to(DEVICE)
        logits = logits.masked_fill(lesion_t <= 0.0, -20.0)
        probs = torch.sigmoid(logits)[0].cpu().numpy()  # (5, H, W)

    masks = (probs >= VGG16_FEATURE_THRESHOLD).astype(np.uint8)
    return {
        'feature_probs_map': probs,
        'feature_masks': masks,
    }


def vgg16_feature_presence(seg_result, min_area_px=VGG16_MIN_AREA_PX):
    """Превращает попиксельные маски в бинарное решение 'есть/нет' для 5 признаков."""
    masks = seg_result['feature_masks']
    presence = np.zeros(VGG16_NUM_CLASSES, dtype=int)
    for i in range(VGG16_NUM_CLASSES):
        area = int(masks[i].sum())
        presence[i] = int(area >= min_area_px)
    return presence


print('✓ Функции VGG16 готовы')

## Grad-CAM

In [ ]:
class GradCAM:
    """Grad-CAM для модели эксп.6.
    target_layer — последний блок backbone EfficientNet.
    """
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(
            lambda m, i, o: setattr(self, 'activations', o.detach()))
        target_layer.register_full_backward_hook(
            lambda m, gi, go: setattr(self, 'gradients', go[0].detach()))

    def generate(self, x, class_idx):
        self.model.zero_grad()
        feat_logits, diag_logit = self.model(x)
        score = feat_logits[0, class_idx] if class_idx < 7 else diag_logit[0, 0]
        score.backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam).squeeze().cpu().numpy()
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam


gradcam = GradCAM(model_exp6, model_exp6.backbone.blocks[-1])


def apply_colormap_on_image(cam, img_np_uint8, alpha=0.5):
    h, w = img_np_uint8.shape[:2]
    cam_resized = cv2.resize(cam, (w, h))
    heatmap = cv2.applyColorMap((cam_resized * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = (alpha * heatmap + (1 - alpha) * img_np_uint8).astype(np.uint8)
    return overlay

print('✓ Grad-CAM готов')

## Rule-based генерация отчёта

In [ ]:
# Клинические описания каждого признака (для текстового отчёта)
FEATURE_DESCRIPTIONS = {
    'pigment_network': 'атипичная пигментная сеть — нерегулярные ячейки и линии тёмного пигмента, размер ячеек варьирует',
    'streaks': 'нерегулярные полосы (streaks) — радиальные линии пигмента по периферии очага',
    'pigmentation': 'нерегулярная пигментация — неоднородное распределение пигмента в очаге',
    'regression_structures': 'структуры регрессии — белые рубцовые участки или серо-голубые гранулы',
    'dots_and_globules': 'нерегулярные точки и глобулы — округлые тёмные структуры разного размера, расположенные хаотично',
    'blue_whitish_veil': 'бело-голубая вуаль — диффузное бело-голубое затенение неровной формы',
    'vascular_structures': 'атипичные сосудистые структуры — точечные или линейные нерегулярные сосуды',
}

def generate_report(feat_probs, feat_preds, diag_prob, vgg_presence=None):
    """Структурированный отчёт с OR-объединением по 3 общим признакам.

    Решение принимается по тем 3 признакам, для которых одновременно доступна
    классификация (моя модель) и сегментация (VGG16 А. Агафьиной). Признак идёт
    в отчёт, если его обнаружила хотя бы одна модель.
    """
    # Применяем OR для 3 общих признаков
    combined_preds = feat_preds.copy()
    if vgg_presence is not None:
        for my_i, vgg_i in COMMON_FEATURES_IDX_MAP.items():
            combined_preds[my_i] = int(bool(feat_preds[my_i]) or bool(vgg_presence[vgg_i]))

    # Балл по 3 общим признакам (на базе объединённых предсказаний)
    common_score = sum(int(combined_preds[i]) * ARGENZIANO_SCORES[i]
                       for i in COMMON_FEATURES_IDX)
    common_suspicious = common_score >= COMMON_SUSPICION_THRESHOLD
    common_found = [i for i in COMMON_FEATURES_IDX if combined_preds[i] == 1]

    # Итоговый вердикт = подозрение по баллу ИЛИ по диагнозу
    diag_suspicious = diag_prob >= CLASSIFIER_THRESHOLD
    final_suspicious = common_suspicious or diag_suspicious

    lines = []
    lines.append('═' * 70)
    lines.append('  ОТЧЁТ ПО АНАЛИЗУ ДЕРМАТОСКОПИЧЕСКОГО ИЗОБРАЖЕНИЯ')
    lines.append('═' * 70)
    lines.append('')

    # ИТОГОВЫЙ ВЕРДИКТ
    if final_suspicious:
        lines.append('  ⚠  ВЕРДИКТ: ТРЕБУЕТСЯ КОНСУЛЬТАЦИЯ ВРАЧА')
    else:
        lines.append('  ✓  ВЕРДИКТ: ВЫРАЖЕННЫХ ПРИЗНАКОВ МЕЛАНОМЫ НЕ ОБНАРУЖЕНО')
    lines.append('')
    lines.append('─' * 70)
    lines.append('')

    # Блок 1: Обнаруженные признаки (только 3 общих)
    lines.append('ОБНАРУЖЕННЫЕ КЛИНИЧЕСКИЕ ПРИЗНАКИ')
    lines.append('')
    if not common_found:
        lines.append('   Из трёх анализируемых признаков ни один не обнаружен.')
    else:
        for i in common_found:
            feat_en = FEATURES[i]
            feat_ru = FEAT_RU[i]
            weight = ARGENZIANO_SCORES[i]
            desc = FEATURE_DESCRIPTIONS.get(feat_en, '')
            star = ' ★' if weight == 2 else ''
            lines.append(f'   ▸ {feat_ru} ({weight} балл{star}) — '
                         f'уверенность {feat_probs[i]*100:.0f}%')
            lines.append(f'     {desc}')
            lines.append('')

    # Признаки на грани (только из 3 общих, не обнаруженные после OR)
    common_low = [(FEAT_RU[i], feat_probs[i]) for i in COMMON_FEATURES_IDX
                  if combined_preds[i] == 0 and feat_probs[i] > 0.25]
    if common_low:
        lines.append('   Дополнительно (на грани порога, рекомендуется проверка врачом):')
        for name, prob in sorted(common_low, key=lambda x: -x[1]):
            lines.append(f'     – {name}: {prob*100:.0f}%')
        lines.append('')

    # Блок 2: Балл и диагноз
    lines.append('─' * 70)
    lines.append('')
    lines.append('ОЦЕНКА')
    lines.append('')
    lines.append(f'   Балл по шкале Аргензиано (по 3 общим признакам): '
                 f'{common_score} из {COMMON_MAX_SCORE}')
    score_status = '⚠ подозрительно' if common_suspicious else '✓ норма'
    lines.append(f'   Порог подозрения: {COMMON_SUSPICION_THRESHOLD}  →  {score_status}')
    lines.append('')
    lines.append(f'   Вероятность меланомы (прямая оценка): {diag_prob*100:.1f}%')
    diag_status = '⚠ подозрительно' if diag_suspicious else '✓ норма'
    lines.append(f'   Порог: {CLASSIFIER_THRESHOLD*100:.0f}%  →  {diag_status}')
    lines.append('')

    # Блок 3: Рекомендация
    lines.append('─' * 70)
    lines.append('')
    if final_suspicious:
        lines.append('РЕКОМЕНДАЦИЯ')
        lines.append('   Рекомендуется консультация дерматолога-онколога.')
        lines.append('   Решение о биопсии и дальнейшей тактике принимает врач.')
    else:
        lines.append('РЕКОМЕНДАЦИЯ')
        lines.append('   Плановое наблюдение в обычном порядке.')
    lines.append('')
    lines.append('═' * 70)
    lines.append('Демонстрационная версия. Решение принимает врач.')
    lines.append('═' * 70)

    return '\n'.join(lines)

print('✓ Генератор отчёта готов')

## Главная функция пайплайна (всё вместе)

In [ ]:
import tempfile

def run_full_pipeline(input_image_pil: Image.Image):
    """Полный пайплайн на одном изображении.
    Возвращает словарь со всеми артефактами для Gradio.
    """
    # Конвертация в RGB и сохранение во временный файл (RT-DETR хочет путь)
    input_image_pil = input_image_pil.convert('RGB')
    with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as tmp:
        input_image_pil.save(tmp.name, 'JPEG')
        tmp_path = Path(tmp.name)

    # Детекция
    det = detect_lesion(input_image_pil, tmp_path)
    x1, y1, x2, y2 = det.box_xyxy
    crop = input_image_pil.crop((x1, y1, x2, y2))

    # Визуализация детекции
    img_np = np.array(input_image_pil)
    det_vis = img_np.copy()
    color_box = (0, 255, 255) if not det.used_fallback else (255, 165, 0)
    cv2.rectangle(det_vis, (x1, y1), (x2, y2), color_box,
                  thickness=max(2, input_image_pil.width // 250))

    # Удаление волос
    hair_result = remove_hair_on_crop(crop)
    crop_320 = (hair_result['crop_01'].permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    clean_crop_320 = (hair_result['clean_crop_01'].permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    hair_mask_vis = (hair_result['hair_mask'].numpy() * 255).astype(np.uint8)

    # Сегментация невуса
    lesion_mask = segment_lesion(hair_result['clean_crop_01'])
    lesion_mask_vis = (lesion_mask.numpy() * 255).astype(np.uint8)

    # Наложение маски невуса на оригинальный кроп
    lesion_overlay = crop_320.copy()
    red_layer = np.zeros_like(lesion_overlay); red_layer[..., 0] = 255
    alpha_mask = 0.35 * (lesion_mask_vis[..., None] > 127)
    lesion_overlay = (lesion_overlay * (1 - alpha_mask) + red_layer * alpha_mask).astype(np.uint8)

    # Классификация
    classifier_input = prepare_classifier_input(crop, lesion_mask)
    cls_result = classify_features(classifier_input)
    feat_probs = cls_result['feat_probs']
    feat_preds = cls_result['feat_preds']
    diag_prob  = cls_result['diag_prob']

    # Сегментация признаков от А. Агафьиной (VGG16)
    # на вход идёт исходный кроп с волосами (crop, а не clean_crop)
    # Маска невуса (lesion_mask) получена выше от BiRefNet, она общая для обеих веток.
    vgg_result = segment_features_vgg16(crop, lesion_mask)
    vgg_presence = vgg16_feature_presence(vgg_result, min_area_px=VGG16_MIN_AREA_PX)

    # Балл и вердикт с учётом OR-объединения по 3 общим признакам
    score, suspicious, found = compute_argenziano_score(feat_preds, vgg_presence)

    # Grad-CAM для каждого предсказанного признака + диагноза
    classifier_input_dev = classifier_input.to(DEVICE)
    crop_300 = np.array(crop.resize((CLASSIFIER_IMG_SIZE, CLASSIFIER_IMG_SIZE), Image.BILINEAR))

    gradcam_panels = []
    for i in range(7):
        x_grad = classifier_input_dev.clone().requires_grad_(True)
        cam = gradcam.generate(x_grad, class_idx=i)
        overlay = apply_colormap_on_image(cam, crop_300, alpha=0.55)
        gradcam_panels.append((FEAT_RU[i], feat_probs[i], feat_preds[i], overlay))

    # Grad-CAM для головы диагноза
    x_grad = classifier_input_dev.clone().requires_grad_(True)
    cam_diag = gradcam.generate(x_grad, class_idx=7)
    diag_overlay = apply_colormap_on_image(cam_diag, crop_300, alpha=0.55)

    # Отчёт
    report = generate_report(feat_probs, feat_preds, diag_prob, vgg_presence)

    return {
        'detection_vis': det_vis,
        'crop_320': crop_320,
        'hair_mask': hair_mask_vis,
        'clean_crop': clean_crop_320,
        'lesion_overlay': lesion_overlay,
        'gradcam_panels': gradcam_panels,
        'diag_overlay': diag_overlay,
        'diag_prob': diag_prob,
        'score': score,
        'suspicious': suspicious,
        'report': report,
        'vgg_presence': vgg_presence,
        'det_used_fallback': det.used_fallback,
        'det_score': det.score,
    }

print('✓ Пайплайн готов')

## Gradio-интерфейс

In [ ]:
!pip uninstall -y Pillow
!pip install -q -U ultralytics transformers segmentation-models-pytorch timm einops kornia gradio==4.44.0 Pillow==9.5.0

In [ ]:
!pip install -q -U gradio

In [ ]:
!pip install --upgrade gradio


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import gradio as gr

def gradio_pipeline(input_img):
    """Обёртка пайплайна под Gradio."""
    if input_img is None:
        return [None]*8 + ['Загрузите изображение для анализа.']
    pil = Image.fromarray(input_img) if isinstance(input_img, np.ndarray) else input_img
    res = run_full_pipeline(pil)

    # Соберём Grad-CAM: 3 общих признака + диагноз = 4 панели в один ряд
    fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
    for slot_i, feat_i in enumerate(COMMON_FEATURES_IDX):
        name, prob, pred, overlay = res['gradcam_panels'][feat_i]
        axes[slot_i].imshow(overlay)
        weight = ARGENZIANO_SCORES[feat_i]
        marker = ' ★' if weight == 2 else ''
        status = '✓' if pred == 1 else '–'
        color = 'red' if pred == 1 else 'gray'
        axes[slot_i].set_title(f'{status} {name} ({weight}б{marker})\nP={prob*100:.0f}%',
                               fontsize=11, color=color)
        axes[slot_i].axis('off')
    # 4-я ячейка — диагноз
    axes[3].imshow(res['diag_overlay'])
    diag_color = 'red' if res['diag_prob'] >= CLASSIFIER_THRESHOLD else 'green'
    axes[3].set_title(f'Диагноз меланомы\nP={res["diag_prob"]*100:.0f}%',
                      fontsize=11, color=diag_color)
    axes[3].axis('off')
    plt.suptitle('Карты внимания модели — признаки, общие с модулем сегментации',
                 fontsize=12, y=1.02)
    plt.tight_layout()
    gradcam_panel_path = '/content/gradio_gradcam.png'
    plt.savefig(gradcam_panel_path, dpi=110, bbox_inches='tight')
    plt.close()

    # Считаем балл с учётом OR-объединения по 3 общим признакам
    feat_preds_arr = np.array([p[2] for p in res['gradcam_panels']])
    vgg_pres = res.get('vgg_presence')
    combined_for_score = feat_preds_arr.copy()
    if vgg_pres is not None:
        for my_i, vgg_i in COMMON_FEATURES_IDX_MAP.items():
            combined_for_score[my_i] = int(bool(feat_preds_arr[my_i]) or bool(vgg_pres[vgg_i]))

    common_score = sum(int(combined_for_score[i]) * ARGENZIANO_SCORES[i]
                       for i in COMMON_FEATURES_IDX)
    common_suspicious = common_score >= COMMON_SUSPICION_THRESHOLD
    diag_suspicious = res['diag_prob'] >= CLASSIFIER_THRESHOLD
    final = common_suspicious or diag_suspicious
    verdict_md = ('⚠ **ТРЕБУЕТСЯ КОНСУЛЬТАЦИЯ ВРАЧА**' if final
                  else '✓ **Выраженных признаков меланомы не обнаружено**')

    # Перегенерируем отчёт с учётом VGG16
    new_report = generate_report(
        feat_probs=np.array([p[1] for p in res['gradcam_panels']]),
        feat_preds=feat_preds_arr,
        diag_prob=res['diag_prob'],
        vgg_presence=vgg_pres,
    )

    return (
        res['detection_vis'],
        res['crop_320'],
        res['hair_mask'],
        res['clean_crop'],
        res['lesion_overlay'],
        gradcam_panel_path,
        f'### {verdict_md}\n\n'
        f'**Балл Аргензиано (по 3 общим признакам):** {common_score} / {COMMON_MAX_SCORE}  \n'
        f'**Порог подозрения:** {COMMON_SUSPICION_THRESHOLD}  \n'
        f'**Вероятность меланомы:** {res["diag_prob"]*100:.1f}%',
        new_report,
    )


with gr.Blocks(title='NevoScan') as demo:
    gr.Markdown('# NevoScan — анализ дерматоскопических изображений')
    gr.Markdown(
        'Загрузите дерматоскопический снимок родинки. Система выполнит детекцию, '
        'удалит волосяные артефакты, выделит область невуса, классифицирует 7 признаков '
        'по шкале Аргензиано и оценит вероятность меланомы. Результаты включают '
        'Grad-CAM-карты внимания и итоговый отчёт.'
    )

    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(label='Дерматоскопический снимок', type='numpy', height=350)
            run_btn = gr.Button('🔬 Запустить анализ', variant='primary', size='lg')
            summary = gr.Markdown(label='Итог')
        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.Tab('Этапы пайплайна'):
                    with gr.Row():
                        det_vis = gr.Image(label='1. Детекция родинки (RT-DETR)', height=250)
                        crop_vis = gr.Image(label='2. Кроп родинки', height=250)
                    with gr.Row():
                        hair_vis = gr.Image(label='3. Маска волос', height=250)
                        clean_vis = gr.Image(label='4. После удаления волос', height=250)
                    lesion_vis = gr.Image(label='5. Маска невуса (BiRefNet)', height=300)
                with gr.Tab('Grad-CAM (карты внимания)'):
                    gradcam_vis = gr.Image(label='Внимание модели по каждому признаку и диагнозу',
                                            height=600)
                with gr.Tab('Текстовый отчёт'):
                    report_box = gr.Textbox(label='Отчёт', lines=35, max_lines=50)

    run_btn.click(
        fn=gradio_pipeline,
        inputs=[input_image],
        outputs=[det_vis, crop_vis, hair_vis, clean_vis, lesion_vis,
                 gradcam_vis, summary, report_box],
    )

    gr.Markdown(
        '---\n'
        '*Демонстрационная версия пайплайна NevoScan. Не заменяет очный осмотр врача.*\n'
        '*Пайплайн объединяет модели сегментации А.Д. Агафьиной и классификационную '
        'модель эксперимента 6 настоящей работы.*'
    )

demo.launch(share=True, theme=gr.themes.Soft(), debug=True)